In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

# SPDFI
# setup
# preprocess (list image, load, smoothing)
# detect (keypoint/descriptor)
# find+filter (match + ratio test)
# inference (draw match + show match)

In [ ]:
# SETUP
target_image_path = "./CV_Session6/Dataset/Object.jpg"
dataset_folder_path = "./CV_Session6/Dataset/Data/"

In [ ]:
# PREPROCESS (ori -> rgb -> gray -> smoothing)

# 1. load target image
# 2. convert to RGB
# 3. convert to grayscale
# 4. apply Gaussian blur to reduce noise

# 5. load dataset images, convert to RGB, convert to grayscale, apply Gaussian blur
# 6. store dataset record as tuple (filename, rgb, gray)

target_bgr_image = cv2.imread(target_image_path)
if target_bgr_image is None:
    raise FileNotFoundError(f"Target image not found: {target_image_path}")

target_rgb_image = cv2.cvtColor(target_bgr_image, cv2.COLOR_BGR2RGB)
target_gray_image = cv2.cvtColor(target_rgb_image, cv2.COLOR_RGB2GRAY)
target_gray_blurred = cv2.GaussianBlur(target_gray_image, (3, 3), 0)

dataset_records = []
for candidate_filename in os.listdir(dataset_folder_path):
    candidate_path = os.path.join(dataset_folder_path, candidate_filename)
    if not os.path.isfile(candidate_path):
        continue

    candidate_bgr_image = cv2.imread(candidate_path)
    if candidate_bgr_image is None:
        continue

    candidate_rgb_image = cv2.cvtColor(candidate_bgr_image, cv2.COLOR_BGR2RGB)
    candidate_gray_image = cv2.cvtColor(candidate_rgb_image, cv2.COLOR_RGB2GRAY)
    candidate_gray_image = cv2.GaussianBlur(candidate_gray_image, (3, 3), 0)
    dataset_records.append((candidate_filename, candidate_rgb_image, candidate_gray_image))

if not dataset_records:
    raise RuntimeError(f"No valid images found in: {dataset_folder_path}")

In [ ]:
# DETECT
akaze_detector = cv2.AKAZE_create()
target_keypoints, target_descriptors = akaze_detector.detectAndCompute(target_gray_blurred, None)

if target_descriptors is None or len(target_descriptors) == 0:
    raise RuntimeError("No descriptors detected on target image.")

In [ ]:
# FIND + FILTER
bf_matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
best_good_match_count = 0
best_match_record = None
candidate_match_scores = []

for candidate_filename, candidate_rgb_image, candidate_gray_image in dataset_records:
    candidate_keypoints, candidate_descriptors = akaze_detector.detectAndCompute(candidate_gray_image, None)

    knn_matches = bf_matcher.knnMatch(target_descriptors, candidate_descriptors, k=2)

    good_match_mask = [[0, 0] for _ in range(len(knn_matches))]
    good_match_count = 0

    for match_index, match_pair in enumerate(knn_matches):
        first_match, second_match = match_pair
        if first_match.distance < 0.7 * second_match.distance:
            good_match_mask[match_index] = [1, 0]
            good_match_count += 1

    candidate_match_scores.append((candidate_filename, good_match_count))

    if best_good_match_count < good_match_count:
        best_good_match_count = good_match_count
        best_match_record = {
            "filename": candidate_filename,
            "rgb_image": candidate_rgb_image,
            "keypoints": candidate_keypoints,
            "descriptors": candidate_descriptors,
            "knn_matches": knn_matches,
            "good_match_mask": good_match_mask
        }

if best_match_record is None:
    raise RuntimeError("No good match found in dataset.")

candidate_match_scores = sorted(candidate_match_scores, key=lambda item: item[1], reverse=True)

In [ ]:
# INFERENCE
matched_visualization = cv2.drawMatchesKnn(
    target_rgb_image,
    target_keypoints,
    best_match_record["rgb_image"],
    best_match_record["keypoints"],
    best_match_record["knn_matches"],
    outImg=None,
    matchesMask=best_match_record["good_match_mask"],
    matchColor=(0, 255, 0),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(16, 8))
plt.imshow(matched_visualization)
plt.title(f"Best: {best_match_record['filename']} | Good Matches: {best_good_match_count}")
plt.axis("off")
plt.show()

In [ ]:
# TOP CANDIDATES
for rank, (candidate_filename, good_match_count) in enumerate(candidate_match_scores[:5], start=1):
    print(f"{rank}. {candidate_filename}: {good_match_count} good matches")